# Task 3 ablation: which of the new pieces actually help?

The new `task3.py` changes seven things at once: tuned GB parameters, 5-fold fitting of the
combiner (`n_folds`), `TopKItemKNN` (with time-weighted means) in place of the Jaccard
`ItemKNN`, `UserKNN`, `RatingProfile`, and `SequenceResidual`. This notebook adds them one
at a time on the outer 10% holdout, then drops each new source from the full model, then
checks the best config against the shipped one over several outer splits.

Reference: the shipped stack (time-bias base; Jaccard + adjusted + raw kNN, stats, session)
scored **0.6908** on the outer holdout. A0 should reproduce that.

The outer holdout is never seen by any source or by the combiner, so it also catches leakage.
If a feature leaked its own row's rating during combiner training (e.g. `SequenceResidual`
seeing the row itself), the combiner would learn to trust it, and outer MSE would get worse,
not better.

In [1]:
import inspect
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

from assignment_1.modules.models import (
    ItemKNN,
    RatingProfile,
    RatingStats,
    SequenceResidual,
    SessionResidual,
    StackedRating,
    TimeBiasModel,
    TopKItemKNN,
    UserKNN,
    adjusted_cosine,
    jaccard,
)
from assignment_1.modules.tasks.task3 import TASK3_GB_PARAMS, TASK3_TIME_PARAMS, TASK3_TRAIN_FILE
from assignment_1.modules.util import drop_zero_ratings, load_csv

# repo root = first parent that has a data/ folder (works from notebooks/ or the root)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir())
DATA_DIR = ROOT / "data"

OUTER_FRAC = 0.10
SEEDS = (0, 1, 2, 3)

# How the combiner is fit. OLD_FIT should give the shipped behaviour (one internal 75/25
# split); set it the same way as in task1_ablation.
OLD_FIT: dict = {}
NEW_FIT = {"n_folds": 5, "n_jobs": 3}

# Shipped GB was sklearn defaults. random_state pins the internal early-stopping split.
DEFAULT_GB = {"random_state": 0}
TIME_SCALES = (14, 90, 365)

train_all = drop_zero_ratings(load_csv(TASK3_TRAIN_FILE, DATA_DIR))
print(inspect.signature(StackedRating))
print(f"{len(train_all):,} nonzero training rows")
print(f"duplicate (user, item) rows: {int(train_all.duplicated(['user', 'item']).sum())}")

(base: assignment_1.modules.models.rating.base.RatingModel, sources: list[assignment_1.modules.models.rating.stacking.FeatureSource], regressor, val_frac: float = 0.25, seed: int = 0, scale: tuple[float, float] = (1, 5), n_folds: int | None = None, n_jobs: int = 1) -> None
1,405,354 nonzero training rows
duplicate (user, item) rows: 0


## Configs

In [2]:
def make_sources(topk: bool = False, user: bool = False, profile: bool = False,
                 sequence: bool = False) -> list:
    """The shipped Task 3 sources, with the new pieces switched on by flag (same order as task3.py)."""
    jac = (TopKItemKNN(similarity=jaccard, prefix="knn_jac", time_scales=TIME_SCALES) if topk
           else ItemKNN(similarity=jaccard, prefix="knn_jac"))
    sources = [
        jac,
        ItemKNN(similarity=adjusted_cosine, prefix="knn_adj"),
        ItemKNN(similarity=jaccard, target="rating", prefix="knn_raw"),
    ]
    if user:
        sources.append(UserKNN())
    sources.append(RatingStats())
    if profile:
        sources.append(RatingProfile())
    sources.append(SessionResidual())
    if sequence:
        sources.append(SequenceResidual())
    return sources


ALL = dict(topk=True, user=True, profile=True, sequence=True)
G, F = TASK3_GB_PARAMS, NEW_FIT

CONFIGS = {
    # cumulative: add one change at a time
    "A0 shipped":           dict(gb=DEFAULT_GB, fit=OLD_FIT, src={}),
    "A1 + tuned GB":        dict(gb=G, fit=OLD_FIT, src={}),
    "A2 + 5 folds":         dict(gb=G, fit=F, src={}),
    "A3 + top-k item kNN":  dict(gb=G, fit=F, src=dict(topk=True)),
    "A4 + user kNN":        dict(gb=G, fit=F, src=dict(topk=True, user=True)),
    "A5 + profile":         dict(gb=G, fit=F, src=dict(topk=True, user=True, profile=True)),
    "A6 + sequence (new)":  dict(gb=G, fit=F, src=ALL),
    # leave one out from the full model (A6 − sequence is A5)
    "A6 − top-k":           dict(gb=G, fit=F, src={**ALL, "topk": False}),
    "A6 − user kNN":        dict(gb=G, fit=F, src={**ALL, "user": False}),
    "A6 − profile":         dict(gb=G, fit=F, src={**ALL, "profile": False}),
}


def build(cfg: dict) -> StackedRating:
    return StackedRating(
        base=TimeBiasModel(**TASK3_TIME_PARAMS),
        sources=make_sources(**cfg["src"]),
        regressor=HistGradientBoostingRegressor(**cfg["gb"]),
        **cfg["fit"],
    )


def outer_split(df: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    hold = df.sample(frac=OUTER_FRAC, random_state=seed)
    return df.drop(hold.index), hold


def evaluate(cfg: dict, seed: int = 0) -> tuple[float, float]:
    """(holdout MSE, seconds) for one config on one outer split."""
    tr, ho = outer_split(train_all, seed)
    t0 = time.time()
    pred = build(cfg).fit(tr).predict(ho)
    mse = float(np.mean((pred - ho["rating"].to_numpy(dtype=float)) ** 2))
    return mse, time.time() - t0

## Ablation on one outer split
Slow: `UserKNN` and 5 folds multiply fit time. Each row prints as it finishes.

In [3]:
results = []
for name, cfg in CONFIGS.items():
    mse, secs = evaluate(cfg, seed=0)
    results.append({"config": name, "mse": mse, "minutes": secs / 60})
    print(f"{name:24s} {mse:.4f}  ({secs / 60:.1f} min)")

table = pd.DataFrame(results).set_index("config")
base = table.loc["A0 shipped", "mse"]
table["vs A0"] = (table["mse"] / base - 1).map(lambda x: f"{x:+.2%}")
table["step"] = table["mse"].diff().map(lambda x: f"{x:+.4f}" if pd.notna(x) else "")
table

A0 shipped               0.6931  (1.8 min)
A1 + tuned GB            0.6926  (1.9 min)
A2 + 5 folds             0.6908  (2.9 min)
A3 + top-k item kNN      0.6833  (4.6 min)
A4 + user kNN            0.6812  (5.4 min)
A5 + profile             0.6775  (5.7 min)
A6 + sequence (new)      0.6760  (6.0 min)
A6 − top-k               0.6818  (4.2 min)
A6 − user kNN            0.6783  (5.4 min)
A6 − profile             0.6799  (5.6 min)


,mse,minutes,vs A0,step
config,,,,
A0 shipped,0.693081,1.829500,+0.00%,
A1 + tuned GB,0.692585,1.916629,-0.07%,-0.0005
A2 + 5 folds,0.690794,2.919418,-0.33%,-0.0018
A3 + top-k item kNN,0.683309,4.607711,-1.41%,-0.0075
A4 + user kNN,0.681185,5.388501,-1.72%,-0.0021
A5 + profile,0.677518,5.678424,-2.25%,-0.0037
A6 + sequence (new),0.676044,5.971023,-2.46%,-0.0015
A6 − top-k,0.681822,4.196221,-1.62%,+0.0058
A6 − user kNN,0.678258,5.427154,-2.14%,-0.0036


**Reading it.** For the cumulative rows (A1–A6), "step" is what that one change bought.
For the leave-one-out rows, compare against A6: if dropping a source costs less than about
0.001, that's within noise and the source isn't earning its keep. Set `BEST` to the smallest
config whose gains are all real, then check it over the other outer splits.

## Seed check: best vs shipped over several outer splits

In [4]:
BEST = "A6 + sequence (new)"   # change after reading the table above

rows = []
for seed in SEEDS:
    a0, _ = evaluate(CONFIGS["A0 shipped"], seed)
    best, _ = evaluate(CONFIGS[BEST], seed)
    rows.append({"seed": seed, "A0": a0, BEST: best, "diff": best - a0})
    print(f"seed {seed}: A0 {a0:.4f}  {BEST} {best:.4f}  diff {best - a0:+.4f}")

seeds = pd.DataFrame(rows).set_index("seed")
print(f"\nmean diff {seeds['diff'].mean():+.4f}, "
      f"better on {(seeds['diff'] < 0).sum()}/{len(seeds)} splits")
seeds

seed 0: A0 0.6931  A6 + sequence (new) 0.6760  diff -0.0170
seed 1: A0 0.6925  A6 + sequence (new) 0.6772  diff -0.0153
seed 2: A0 0.6938  A6 + sequence (new) 0.6774  diff -0.0164
seed 3: A0 0.6911  A6 + sequence (new) 0.6746  diff -0.0165

mean diff -0.0163, better on 4/4 splits


,A0,A6 + sequence (new),diff
seed,,,
0,0.693081,0.676044,-0.017037
1,0.692502,0.677210,-0.015292
2,0.693833,0.677407,-0.016426
3,0.691099,0.674609,-0.016489
